# REUSABLE PIPELINE
A reusable pipeline is a structured workflow where each step is modular, configurable, and reusable across different projects or datasets.

In [120]:
import pandas as pd
import numpy as np
from io import StringIO
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score


In [121]:
df=pd.read_csv("cs_students.csv")
df

,Student ID,Name,Gender,Age,GPA,Major,Interested Domain,Projects,Future Career,Python,SQL,Java
0,1,John Smith,Male,21,3.5,Computer Science,Artificial Intelligence,Chatbot Development,Machine Learning Researcher,Strong,Strong,Weak
1,2,Alice Johnson,Female,20,3.2,Computer Science,Data Science,Data Analytics,Data Scientist,Average,Strong,Weak
2,3,Robert Davis,Male,22,3.8,Computer Science,Software Development,E-commerce Website,Software Engineer,Strong,Strong,Average
3,4,Emily Wilson,Female,21,3.7,Computer Science,Web Development,Full-Stack Web App,Web Developer,Weak,Strong,Strong
4,5,Michael Brown,Male,23,3.4,Computer Science,Cybersecurity,Network Security,Information Security Analyst,Average,Weak,Strong
...,...,...,...,...,...,...,...,...,...,...,...,...
175,176,Elijah Davis,Male,22,3.7,Computer Science,Web Development,Full-Stack Web App,Web Developer,Weak,Strong,Strong
176,177,Emma Johnson,Female,20,3.6,Computer Science,Cybersecurity,Security Auditing,Information Security Analyst,Strong,Average,Weak
177,178,Liam Wilson,Male,21,3.4,Computer Science,Machine Learning,Natural Language Processing,Machine Learning Engineer,Strong,Average,Weak
178,179,Sophia Johnson,Female,22,3.5,Computer Science,Database Management,SQL Database Administration,Database Administrator,Weak,Strong,Average


In [122]:
def preprocess_data(df):
    df=df.copy()
    skill_map={"Weak": 0, "Average": 1, "Strong": 2}
    for col in ["Python", "SQL", "Java"]:
        df[col] = df[col].map(skill_map)
    df["GPA"] = df["GPA"].fillna(df["GPA"].mean())
    df["Gender"] = df["Gender"].map({"Male": 0, "Female": 1})
    df = df.drop(columns=["Student ID", "Name", "Major", "Projects"])
    le=LabelEncoder()
    df["Interested Domain"] = le.fit_transform(df["Interested Domain"])
    df["Future Career"] = le.fit_transform(df["Future Career"])
    return df


In [123]:
processed_df = preprocess_data(df)
print(processed_df.head())

   Gender  Age  GPA  Interested Domain  Future Career  Python  SQL  Java
0       0   21  3.5                  0             21       2    2     0
1       1   20  3.2                 10              7       1    2     0
2       0   22  3.8                 24             29       2    2     1
3       1   21  3.7                 26             32       0    2     2
4       0   23  3.4                  7             18       1    0     2


In [124]:
X = processed_df.drop(columns=["Future Career"])
y = processed_df["Future Career"]

In [125]:
print(f"Features (X): {list(X.columns)}")
print(f"Target (y): Future Career\n")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = RandomForestClassifier(n_estimators=100, random_state=42)
print(f"   Training samples: {len(X_train)}")
print(f"   Testing  samples: {len(X_test)}\n")
model.fit(X_train, y_train)
y_pred = model.predict(X_test)




Features (X): ['Gender', 'Age', 'GPA', 'Interested Domain', 'Python', 'SQL', 'Java']
Target (y): Future Career

   Training samples: 144
   Testing  samples: 36



In [126]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),           # Step 1: Normalize
    ("model", RandomForestClassifier(       # Step 2: Train model
        n_estimators=100,
        random_state=42
    ))
])

In [127]:
print("✅ Pipeline created with steps:")
for step_name, step_obj in pipeline.steps:
    print(f"   → {step_name}: {type(step_obj).__name__}")


✅ Pipeline created with steps:
   → scaler: StandardScaler
   → model: RandomForestClassifier


In [128]:
pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"✅ Model Accuracy: {accuracy * 100:.1f}%\n")

✅ Model Accuracy: 72.2%



# This is the POWER of reusable pipelines.
Just pass in new student data — the pipeline handles everything!


In [129]:
new_student_raw = pd.DataFrame({
    "Student ID": [999],
    "Name":       ["New Student"],
    "Gender":     ["Male"],
    "Age":        [21],
    "GPA":        [3.8],
    "Major":      ["Computer Science"],
    "Interested Domain": ["Machine Learning"],
    "Projects":   ["AI Chatbot"],
    "Future Career": ["Unknown"],   
    "Python":     ["Strong"],
    "SQL":        ["Weak"],
    "Java":       ["Weak"]
})

In [130]:
print(" New Student Profile:")
print(f"   Name: {new_student_raw['Name'][0]}")
print(f"   GPA: {new_student_raw['GPA'][0]}")
print(f"   Interested Domain: {new_student_raw['Interested Domain'][0]}")
print(f"   Python: {new_student_raw['Python'][0]}, SQL: {new_student_raw['SQL'][0]}, Java: {new_student_raw['Java'][0]}")

 New Student Profile:
   Name: New Student
   GPA: 3.8
   Interested Domain: Machine Learning
   Python: Strong, SQL: Weak, Java: Weak


In [131]:
new_student_processed = preprocess_data(new_student_raw)
X_new = new_student_processed.drop(columns=["Future Career"])

In [133]:
prediction = pipeline.predict(X_new)
le=LabelEncoder()
le.fit(df["Future Career"])
predicted_career = le.inverse_transform(prediction)[0]# Get the original label from the encoded prediction
print(f" Predicted Future Career for New Student: {predicted_career}")



 Predicted Future Career for New Student: NLP Research Scientist
